## Task 5: Inverse Problem

**Goal:** Use a trained regression model and an optimiser to solve the inverse problem — given a target output value `y*`, find the input `X` such that `model(X) ≈ y*`.

1. Load the `diabetes` dataset and split into train / val / test:

```python
from sklearn.datasets import load_diabetes

diabetes      = load_diabetes()
X, y          = diabetes.data.astype(np.float32), diabetes.target.astype(np.float32)
feature_names = diabetes.feature_names
```

2. Implement, train and evaluate a regression model on the diabetes dataset.
3. Implement inverse prediction using `scipy.optimize.minimize`. Try `n_trials` random starting points within training-data bounds and return the best result:

```python
from scipy.optimize import minimize

def inverse_prediction_minimize(model, y_target, X_train, method="L-BFGS-B", n_trials=5):
    """
    Find X such that model(X) ≈ y_target.
    Parameters
    ----------
    model    : trained regression model
    y_target : float, target output value
    X_train  : ndarray, training data used to determine feature bounds
    method   : optimisation algorithm
    n_trials : number of random restarts
    Returns
    -------
    best_x    : ndarray, best found input vector
    best_rmse : float, RMSE at best_x
    """
    def objective(x):
        # compute y_pred using the regressor model and return (y_pred - y_target)**2
        ...

    x_min  = X_train.min(axis=0)
    x_max  = X_train.max(axis=0)
    bounds = list(zip(x_min, x_max))

    best_x, best_error = None, float("inf")
    for trial in range(n_trials):
        x0     = ...  # random starting point within bounds
        result = minimize(objective, x0, method=method, bounds=bounds,
                          options={"maxiter": 1000})
        if result.success and result.fun < best_error:
            best_x, best_error = result.x, result.fun

    return best_x, np.sqrt(best_error)
```

4. Test on the test set:

```python
y_target         = np.median(y_test)
x_inverse, error = inverse_prediction_minimize(model, y_target, X_train, n_trials=10)
y_check          = model(torch.from_numpy(x_inverse.astype(np.float32)).unsqueeze(0).to(DEVICE)).item()
```

5. Propose and implement an Optuna-based approach to solve the inverse problem. Each trial suggests a candidate `X` within training-data bounds; the objective is the squared prediction error.

**Assignment:** Implement inverse prediction based on a trained regression model.

In [1]:
IMBALANCE_RATIO = 0.95

BATCH_SIZE   = 128
EPOCHS       = 1000
LR           = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42

In [2]:
from sklearn.datasets import load_diabetes
import numpy as np

diabetes      = load_diabetes()
X, y          = diabetes.data.astype(np.float32), diabetes.target.astype(np.float32)
feature_names = diabetes.feature_names

In [3]:
import torch
import torch.nn as nn
from utils import make_loaders, train_baseline, BaseRegressionNet
from sklearn.model_selection import train_test_split

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

train_loader, val_loader, test_loader = make_loaders(X, y, batch_size=BATCH_SIZE)
X_train = train_test_split(X, y, test_size=0.2)[0]

diab_model = train_baseline(BaseRegressionNet(in_features=10).to(device),
                            train_loader, val_loader, criterion=nn.MSELoss(),
                            device=device, weight_decay=WEIGHT_DECAY, epochs=EPOCHS, lr=LR)

Epoch 100/1000  train=26443.4328  val=23228.3008
Epoch 200/1000  train=18302.5515  val=16587.7148
Epoch 300/1000  train=10609.4343  val=11573.3965
Epoch 400/1000  train=5876.6661  val=8600.6943
Epoch 500/1000  train=3069.1736  val=6214.6855
Epoch 600/1000  train=1772.6753  val=5290.0420
Epoch 700/1000  train=1332.4730  val=5633.8115
Epoch 800/1000  train=1314.1647  val=5400.8892
Epoch 900/1000  train=1341.0927  val=5266.8032
Epoch 1000/1000  train=983.3290  val=5772.5674


In [4]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

diab_model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = diab_model(X_batch.to(device)).cpu().numpy().flatten()
        y_pred.extend(preds)
        y_true.extend(y_batch.numpy().flatten())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
print(f"MSE: {mean_squared_error(y_true, y_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_true, y_pred):.4f}")
print(f"R2:  {r2_score(y_true, y_pred):.4f}")

MSE: 4006.7678
MAE: 47.4254
R2:  0.1761


In [5]:
from utils import inverse_prediction_minimize, inverse_prediction_optuna

y_target = float(np.median(y_true))

diab_model.eval()
x_scipy,  err_scipy  = inverse_prediction_minimize(diab_model, y_target, X_train, device=device, n_trials=10)
x_optuna, err_optuna = inverse_prediction_optuna(diab_model, y_target, X_train, device=device,  n_trials=200)

y_scipy  = diab_model(torch.from_numpy(x_scipy.astype(np.float32)).unsqueeze(0).to(device)).item()
y_optuna = diab_model(torch.from_numpy(x_optuna).unsqueeze(0).to(device)).item()

print(f"{'':20} {'Scipy':>10} {'Optuna':>10}")
print("=" * 42)
print(f"{'Target y':<20} {y_target:>10.2f} {y_target:>10.2f}")
print(f"{'Predicted y':<20} {y_scipy:>10.2f} {y_optuna:>10.2f}")
print(f"{'RMSE':<20} {err_scipy:>10.4f} {err_optuna:>10.4f}")

                          Scipy     Optuna
Target y                 147.00     147.00
Predicted y              147.00     146.94
RMSE                     0.0000     0.0649
